# Short-term (single-mission) energy evaluation

Notebook version of `python -m evaluation.short_term`. It reuses the same functions, so the numbers match the CLI.

* **By distance**: SOH and initial SOC are fixed, and the mission distance is swept.
* **By SOH**: distance and initial SOC are fixed, and the SOH level is swept.

Every policy sees the same seeded complexity disturbance (common random numbers).
A method with several seed checkpoints is reported per seed (`<method> s<k>`) and as `<method>`, the mean over seeds (std = spread across seeds).

Run this notebook from `rl_train/daro_sac/`.

In [ ]:
import os, glob, json
from pathlib import Path

# Make relative paths (./checkpoints, ./logs) resolve from rl_train/daro_sac
if not Path('evaluation').is_dir():
    for cand in [Path.cwd() / 'rl_train' / 'daro_sac', Path.cwd().parent]:
        if (cand / 'evaluation').is_dir():
            os.chdir(cand)
            break
print('cwd:', os.getcwd())

import numpy as np
import pandas as pd
from IPython.display import Image, display

import evaluation  # sets sys.path (daro_sac + battery_model)
from evaluation import EVAL_CONFIG, make_baselines, load_rl_policies
from evaluation.short_term import run, add_group_means, print_table, write_csv, plot, METRICS

%matplotlib inline

## Settings

In [ ]:
# Checkpoints: DARO and ablations mixed; labels come from each obs_variant.json
CKPT_GROUPS = ['.', 'rl_soc', 'rl_soh', 'rw_no_stress', 'rl_soc_rw_no_stress']
CKPTS = sorted(os.path.normpath(p) for g in CKPT_GROUPS
               for p in glob.glob(os.path.join('checkpoints', g, 'seed_*'))
               if os.path.isfile(os.path.join(p, 'daro_checkpoint.pt')))

EPISODES   = 10                                  # seeds per cell
COMPLEXITY = EVAL_CONFIG['test_complexity_levels']  # [1, 2, 3]
LOG_DIR    = './logs/eval_short_term'

# --by distance
DISTANCES  = [1000.0]
SOH_FIXED  = 1.0
# --by soh
SOH_LEVELS = EVAL_CONFIG['test_soh_levels']      # [1.0, 0.95, 0.9, 0.85, 0.8]
DIST_FIXED = EVAL_CONFIG['mission_distance_m']   # 500 m
# both
SOC0       = 1.0

PLOT_ORDER = ['DARO', 'RL-SOC-only', 'RL-SOH-only', 'DARO w/o D_d', 'DARO w/o E_d']
os.makedirs(LOG_DIR, exist_ok=True)
print(len(CKPTS), 'checkpoints:'); print('\n'.join('  ' + c for c in CKPTS))

## Helpers

In [ ]:
def base_config(distance, soh, soc):
    cfg = dict(EVAL_CONFIG, test_soh_levels=[soh], test_soc_levels=[soc],
               mission_distance_m=distance)
    cfg['test_complexity_levels'] = list(COMPLEXITY)
    return cfg


def load_policies(cfg):
    policies, groups = load_rl_policies(CKPTS, cfg) if CKPTS else ({}, {})
    policies.update(make_baselines(cfg))
    return policies, groups


def experiment(by):
    if by == 'distance':
        cfg = base_config(DIST_FIXED, SOH_FIXED, SOC0)
        values, subtitle = DISTANCES, f', SOH {SOH_FIXED:.2f}, SOC0 {SOC0:.2f}'
    else:
        cfg = base_config(DIST_FIXED, SOH_LEVELS[0], SOC0)
        values, subtitle = SOH_LEVELS, f', {DIST_FIXED:.0f} m, SOC0 {SOC0:.2f}'
    policies, groups = load_policies(cfg)
    print('Policies:', list(policies))

    table = run(policies, cfg, values, EPISODES, by)
    add_group_means(table, groups)
    print_table(table, by=by)

    base = os.path.join(LOG_DIR, 'short_term_energy' + ('' if by == 'distance' else f'_{by}'))
    with open(base + '.json', 'w') as f:
        json.dump({'config': cfg, 'args': dict(by=by, ckpts=CKPTS, episodes=EPISODES,
                                               values=list(values), soc=SOC0),
                   'table': {str(d): {str(c): row for c, row in per_c.items()}
                             for d, per_c in table.items()}}, f, indent=2, default=float)
    write_csv(table, base + '.csv', by)
    show = (sorted(groups, key=lambda g: PLOT_ORDER.index(g) if g in PLOT_ORDER else 9)
            + list(make_baselines(cfg)))
    plot(table, base + '.png', show, by, subtitle)
    plot(table, base.replace('short_term_energy', 'short_term_qos') + '.png', show, by,
         subtitle, metric='qos_viol_pct')
    return table, groups, base


def to_frame(table, by, ref='AS-MF'):
    """Long DataFrame, one row per (value, complexity, policy), with dE vs ref [%]."""
    col = 'distance_m' if by == 'distance' else 'soh'
    rows = []
    for d, per_c in table.items():
        for c, row in per_c.items():
            e_ref = row[ref]['energy_kj'] if ref in row else np.nan
            for p, m in row.items():
                r = {col: d, 'complexity': c, 'policy': p}
                r.update({k: m.get(k, np.nan) for k in METRICS})
                r['energy_kj_std'] = m.get('energy_kj_std', np.nan)
                r['qos_viol_pct_std'] = m.get('qos_viol_pct_std', np.nan)
                r[f'dE_vs_{ref}_%'] = 100 * (m['energy_kj'] / e_ref - 1)
                rows.append(r)
    return pd.DataFrame(rows)


def method_summary(df, groups, by, value='energy_kj'):
    """Seed-averaged methods + baselines only (drops '<method> s<k>' rows)."""
    col = 'distance_m' if by == 'distance' else 'soh'
    seeds = {n for names in groups.values() if len(names) > 1 for n in names}
    out = df[~df.policy.isin(seeds)]
    return out.pivot_table(index=[col, 'complexity'], columns='policy',
                           values=value, sort=False).round(3)

## 1. Energy vs complexity × mission distance

In [ ]:
table_d, groups_d, base_d = experiment('distance')
df_d = to_frame(table_d, 'distance')
display(Image(base_d + '.png'))

In [ ]:
print('Energy per mission [kJ], seed means:')
display(method_summary(df_d, groups_d, 'distance'))
df_d.round(3)

## 2. Energy vs complexity × SOH

In [ ]:
table_s, groups_s, base_s = experiment('soh')
df_s = to_frame(table_s, 'soh')
display(Image(base_s + '.png'))

In [ ]:
print('Energy per mission [kJ], seed means:')
display(method_summary(df_s, groups_s, 'soh'))
df_s.round(3)

## 2b. QoS violation vs complexity × SOH

QoS violation is the share of control steps where the delivered QoS is below the reference (from `metrics.run_episode`). Lower is better.

In [ ]:
display(Image(base_s.replace('short_term_energy', 'short_term_qos') + '.png'))
print('QoS violation [% of steps], seed means:')
display(method_summary(df_s, groups_s, 'soh', value='qos_viol_pct').round(1))

## 3. Energy saving vs AS-MF (seed means)

In [ ]:
for name, df, groups, by in [('distance', df_d, groups_d, 'distance'), ('SOH', df_s, groups_s, 'soh')]:
    col = 'distance_m' if by == 'distance' else 'soh'
    seeds = {n for names in groups.values() if len(names) > 1 for n in names}
    t = (df[~df.policy.isin(seeds)]
         .pivot_table(index=[col, 'complexity'], columns='policy', values='dE_vs_AS-MF_%', sort=False)
         .round(1))
    print(f'dE vs AS-MF [%] (by {name}); negative = less energy')
    display(t)